# 06 — TTF-Storage Correlation Analysis
Explores how EU gas storage fill % and TTF M1 price co-move across time, seasons, and tenors.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f"Project root: {_c}")
        break

DATA = Path("data/processed")
RAW  = Path("data/raw")

# ── Storage data ──────────────────────────────────────────────────────────────
storage = pd.read_parquet(DATA / "eu_aggregate_full.parquet")
storage.index = pd.to_datetime(storage.index).tz_localize(None)
storage = storage.sort_index()

# ── TTF curve ─────────────────────────────────────────────────────────────────
ttf = pd.read_csv(RAW / "ttf_curve.csv", index_col=0, parse_dates=True)
ttf.index = pd.to_datetime(ttf.index).tz_localize(None)
ttf = ttf.sort_index()

# ── Joined on common dates ────────────────────────────────────────────────────
df = (
    storage[["full", "gasInStorage", "injection", "withdrawal"]]
    .join(ttf[["M1", "M2", "M3", "M6", "M12"]].rename(columns=lambda c: f"TTF_{c}"), how="inner")
    .dropna(subset=["full", "TTF_M1"])
)
df["year"]   = df.index.year
df["month"]  = df.index.month
df["season"] = df["month"].map(
    lambda m: "Injection (Apr-Oct)" if 4 <= m <= 10 else "Withdrawal (Nov-Mar)"
)

print(f"Storage rows:  {len(storage)}")
print(f"TTF rows:      {len(ttf)}")
print(f"Joined rows:   {len(df)}  ({df.index[0].date()} -> {df.index[-1].date()})")
print(f"Columns: {list(df.columns)}")


## 1. Rolling Pearson Correlation
30d, 60d, and 90d rolling correlations between EU fill % and TTF M1.

In [ ]:
windows = {"30d": 30, "60d": 60, "90d": 90}
colors  = {"30d": "#95A5A6", "60d": "#2E75B6", "90d": "#E67E22"}

fig = go.Figure()
for label, w in windows.items():
    rc = df["full"].rolling(w).corr(df["TTF_M1"])
    fig.add_trace(go.Scatter(
        x=rc.index, y=rc, name=f"{label} rolling",
        line=dict(color=colors[label], width=1.6 if label == "60d" else 1.1)
    ))

fig.add_hline(y=0,    line_dash="dot", line_color="black", line_width=0.8)
fig.add_hline(y=-0.7, line_dash="dash", line_color="red", line_width=1,
              annotation_text="-0.7 (storage dominant)", annotation_position="bottom right")
fig.add_hline(y=0.7,  line_dash="dash", line_color="green", line_width=1,
              annotation_text="+0.7", annotation_position="top right")
fig.update_layout(
    title="Rolling Pearson Correlation: EU Fill % vs TTF M1 Price",
    xaxis_title="Date", yaxis_title="Correlation coefficient",
    yaxis=dict(range=[-1.1, 1.1]),
    height=440, template="plotly_white"
)
fig.show()

corr_60d = float(df["full"].rolling(60).corr(df["TTF_M1"]).dropna().iloc[-1])
if corr_60d < -0.7:
    interp = "storage dominant: higher fill -> lower price"
elif corr_60d < -0.3:
    interp = "moderate negative: fill exerts downward pressure on price"
elif corr_60d < 0.3:
    interp = "near zero: no clear fill-price relationship currently"
else:
    interp = "positive: price and fill moving together (demand driven)"
print(f"Current 60d rolling correlation: {corr_60d:.3f}")
print(f"Interpretation: {interp}")


## 2. OLS Regression: log(TTF M1) ~ Fill %
Log-linear regression to quantify price sensitivity to storage levels.

In [ ]:
from scipy import stats

mask = df["full"].notna() & df["TTF_M1"].notna() & (df["TTF_M1"] > 0)
x = df.loc[mask, "full"].values
y = np.log(df.loc[mask, "TTF_M1"].values)

slope, intercept, r, p_val, se = stats.linregress(x, y)
r2 = r ** 2

print(f"OLS: log(TTF M1) ~ fill%")
print(f"{'='*45}")
print(f"  Slope:     {slope:.4f}  (log-price per pp fill)")
print(f"  Intercept: {intercept:.4f}")
print(f"  R²:        {r2:.3f}")
print(f"  p-value:   {p_val:.2e}")
print(f"  Std error: {se:.5f}")
print()

# Price sensitivity table at key fill levels
print(f"  Price sensitivity at fill levels:")
print(f"  {'Fill':>8}  {'Pred TTF M1':>12}  {'% change vs 50%':>17}")
print(f"  {'-'*40}")
p50_pred = np.exp(intercept + slope * 50)
for fill in [20, 30, 40, 50, 60, 70, 80, 90, 100]:
    pred = np.exp(intercept + slope * fill)
    pct_chg = (pred / p50_pred - 1) * 100
    print(f"  {fill:>7}%  {pred:>11.2f}  {pct_chg:>+16.1f}%")

# Scatter with regression line
fill_grid = np.linspace(x.min(), x.max(), 200)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=np.exp(y), mode="markers",
    marker=dict(color=df.loc[mask, "year"], colorscale="Viridis",
                size=3, opacity=0.5, colorbar=dict(title="Year")),
    name="Observations"
))
fig.add_trace(go.Scatter(
    x=fill_grid, y=np.exp(intercept + slope * fill_grid),
    name=f"OLS fit (R²={r2:.2f})",
    line=dict(color="#E74C3C", width=2.5)
))
fig.update_layout(
    title="EU Fill % vs TTF M1 — OLS Regression (log-linear)",
    xaxis_title="EU Storage Fill (%)", yaxis_title="TTF M1 (EUR/MWh)",
    height=440, template="plotly_white"
)
fig.show()


## 3. Correlation by Calendar Year
Average 60d rolling correlation per year — reveals regime shifts (e.g. 2022 energy crisis).

In [ ]:
corr_by_year = (
    df.groupby("year")
    .apply(lambda g: g["full"].rolling(60, min_periods=20).corr(g["TTF_M1"]).mean())
    .rename("avg_60d_corr")
    .dropna()
)

bar_colors = ["#E74C3C" if c < -0.5 else "#2E75B6" if c < 0 else "#27AE60"
              for c in corr_by_year]

fig = go.Figure(go.Bar(
    x=corr_by_year.index.astype(str),
    y=corr_by_year.values,
    marker_color=bar_colors
))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.add_hline(y=-0.5, line_dash="dash", line_color="red", line_width=1,
              annotation_text="Storage dominant threshold")
fig.update_layout(
    title="Average 60d Rolling Correlation by Year: EU Fill % vs TTF M1",
    xaxis_title="Year", yaxis_title="Avg 60d Correlation",
    yaxis=dict(range=[-1.1, 1.1]),
    height=420, template="plotly_white"
)
fig.show()
print(corr_by_year.round(3).to_string())


## 4. Correlation by Season
Injection season (Apr–Oct) vs withdrawal season (Nov–Mar) — box plots of the 60d rolling correlation distribution.

In [ ]:
rc60 = df["full"].rolling(60, min_periods=20).corr(df["TTF_M1"]).rename("rc60")
df_rc = df.join(rc60).dropna(subset=["rc60"])

# Box plot of 60d rolling correlation split by season
inj_rc = df_rc[df_rc["season"] == "Injection (Apr-Oct)"]["rc60"]
wd_rc  = df_rc[df_rc["season"] == "Withdrawal (Nov-Mar)"]["rc60"]

fig = go.Figure()
fig.add_trace(go.Box(
    y=inj_rc, name="Injection Season (Apr-Oct)",
    marker_color="#27AE60", boxmean="sd"
))
fig.add_trace(go.Box(
    y=wd_rc, name="Withdrawal Season (Nov-Mar)",
    marker_color="#2E75B6", boxmean="sd"
))
fig.add_hline(y=0, line_dash="dot", line_color="black", line_width=0.8)
fig.update_layout(
    title="60d Rolling Correlation by Season: Fill % vs TTF M1",
    yaxis=dict(title="Correlation coefficient", range=[-1.1, 1.1]),
    height=420, template="plotly_white"
)
fig.show()

print(f"Injection season  — median: {inj_rc.median():.3f}  mean: {inj_rc.mean():.3f}  std: {inj_rc.std():.3f}")
print(f"Withdrawal season — median: {wd_rc.median():.3f}  mean: {wd_rc.mean():.3f}  std: {wd_rc.std():.3f}")
print()
print("Interpretation:")
print(f"  Injection (Apr-Oct): fill builds as market prices in supply adequacy.")
print(f"  Withdrawal (Nov-Mar): demand drives both drawdown and prices upward together.")


## 5. Lead-Lag Analysis
Cross-correlation at lags −10 to +10 days between daily changes in fill % and TTF M1 — identifies whether storage leads or lags prices.

In [ ]:
# First-differences: change in fill% and change in TTF M1
df_diff = df[["full", "TTF_M1"]].diff().dropna()

lags  = range(-10, 11)
xcorr = []
for lag in lags:
    if lag == 0:
        corr = df_diff["full"].corr(df_diff["TTF_M1"])
    elif lag > 0:
        # positive lag: fill% leads TTF by `lag` days
        corr = df_diff["full"].corr(df_diff["TTF_M1"].shift(-lag))
    else:
        # negative lag: TTF leads fill% by `abs(lag)` days
        corr = df_diff["full"].corr(df_diff["TTF_M1"].shift(abs(lag)))
    xcorr.append(corr)

xcorr = pd.Series(xcorr, index=lags)
peak_lag = int(xcorr.abs().idxmax())

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(lags), y=xcorr.values,
    marker_color=["#E74C3C" if abs(l) == abs(peak_lag) else "#2E75B6" for l in lags]
))
fig.add_vline(x=0, line_dash="dot", line_color="black", line_width=1)
fig.update_layout(
    title="Lead-Lag Cross-Correlation: Fill% Change vs TTF M1 Change (daily diffs)",
    xaxis=dict(title="Lag (days) — positive = fill% leads TTF", tickvals=list(range(-10,11))),
    yaxis=dict(title="Pearson correlation", range=[-0.5, 0.5]),
    height=420, template="plotly_white"
)
fig.show()

print(f"Peak correlation: {xcorr[peak_lag]:.3f} at lag = {peak_lag} days")
if peak_lag > 0:
    print(f"  => Fill% LEADS TTF price by ~{peak_lag} day(s)")
elif peak_lag < 0:
    print(f"  => TTF price LEADS fill% by ~{abs(peak_lag)} day(s)")
else:
    print(f"  => Contemporaneous (no lead/lag)")
print()
print("Cross-correlations by lag:")
print(xcorr.round(4).to_string())


## 6. 3D Scatter: Fill % × Tenor × Price
Visualises how the fill-price relationship varies across M1–M6 contract tenors over time.

In [ ]:
# 3D: EU fill% x contract tenor (M1-M6) x TTF price
# Stack the M1-M6 columns of the TTF curve together
tenor_cols = [c for c in ["TTF_M1","TTF_M2","TTF_M3","TTF_M6"] if c in df.columns]
stacked = []
for col in tenor_cols:
    m = int(col.replace("TTF_M",""))
    tmp = df[["full","year",col]].rename(columns={col: "price"}).copy()
    tmp["tenor"] = m
    stacked.append(tmp.dropna(subset=["price"]))
stacked_df = pd.concat(stacked, ignore_index=True)

fig = go.Figure(go.Scatter3d(
    x=stacked_df["full"],
    y=stacked_df["tenor"],
    z=stacked_df["price"],
    mode="markers",
    marker=dict(
        size=1.5,
        color=stacked_df["year"],
        colorscale="Viridis",
        opacity=0.5,
        colorbar=dict(title="Year")
    )
))
fig.update_layout(
    title="3D Scatter: EU Fill % × Contract Tenor × TTF Price",
    scene=dict(
        xaxis_title="EU Fill (%)",
        yaxis_title="Tenor (months ahead)",
        zaxis_title="TTF Price (EUR/MWh)"
    ),
    height=580
)
fig.show()

# Quick correlation summary by tenor
print("Pearson correlation (fill% vs price) by tenor:")
for col in tenor_cols:
    m = col.replace("TTF_","")
    c = df["full"].corr(df[col])
    print(f"  {m}: {c:.3f}")
